# Cube Only与MIX模式Matmul开发

## 概述

pyasc Matmul有两种执行模式：**Cube Only模式**仅使用Cube计算单元（AIC核），适用于纯矩阵乘法；**MIX模式**（默认）让AIC和AIV成对协同工作，支持Matmul与Vector算子融合。本节将分别实现两种模式的多核Matmul算子，并深入讲解多核偏移计算和选型策略。

### 学习目标

1. 理解Cube Only模式的概念和`@asc.jit(matmul_cube_only=True)`的使用
2. 理解MIX模式的AIC+AIV协同架构和核数`//2`机制
3. 实现带Bias的Cube Only多核Matmul
4. 实现MIX模式多核Matmul
5. 深入理解`calc_offsets`多核偏移计算
6. 掌握Cube Only vs MIX的选型原则

In [ ]:
# 环境初始化
!mkdir -p Sources/04.04

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("环境初始化完成")

---
# 1. Cube Only模式概念

### 什么是Cube Only模式？

Cube Only模式告诉编译器：这个核函数**只做矩阵乘法**，不需要Vector计算单元参与。编译器据此进行优化，将代码部署到AIC核上。

### 启用方式

```python
@asc.jit(matmul_cube_only=True, always_compile=True)
def matmul_kernel(...):
    if asc.ascend_is_aic():      # 仅在AIC核上执行
        # Cube计算逻辑
        ...
```

### 关键特性

| 特性 | 说明 |
| --- | --- |
| 装饰器 | `@asc.jit(matmul_cube_only=True)` |
| 核函数守卫 | `if asc.ascend_is_aic():` 包裹计算逻辑 |
| Launch核数 | `tiling.used_core_num`（不除以2） |
| Bias支持 | 通过`set_bias`设置 |
| set_tail | 传入3个参数 `set_tail(tail_m, tail_n, tiling.k_a)`（同一API，tail_k默认-1） |
| 适用场景 | 纯Matmul（无融合算子）、追求Cube利用率最大化 |

---
# 2. MIX模式概念

### AIC+AIV协同架构

每个AI Core包含两类计算单元：
- **AIC（AI Core）**：执行Cube计算（mmad指令）
- **AIV（AI Vector）**：执行Vector计算（add/relu等指令）

在MIX模式下，AIC和AIV成对工作。核函数启动时，每个"核组"包含1个AIC和1个AIV，因此实际启动的核组数为总核数的一半。

```
总核数: 48 (24个AIC + 24个AIV)
MIX模式启动核组数: 48 // 2 = 24
每个核组: 1个AIC做Cube计算 + 1个AIV做Vector计算
```

<img src="./images/04.04_cube_only_and_mix_mode_matmul/aic_aiv_pairing.png" alt="AIC+AIV配对对比" width="700px">

Cube Only模式独占全部AIC核，每个核独立处理一个数据分片，核数等于`used_core_num`；MIX模式将AIC和AIV配对为核组，每个核组中AIC负责Cube计算、AIV负责Vector计算，核组数为`used_core_num // 2`。对于纯Matmul场景，Cube Only能获得更高的Cube利用率；当需要Matmul与Vector算子融合时（如Matmul+LeakyReLU），MIX模式可避免中间数据回写GM，提升整体性能。

---
# 3. Cube Only与MIX模式对比

| 维度 | Cube Only模式 | MIX模式 |
| --- | --- | --- |
| **装饰器** | `matmul_cube_only=True` | 默认（不加该参数） |
| **使用核类型** | 仅AIC核 | AIC + AIV核 |
| **Launch核数** | `used_core_num` | `used_core_num // 2` |
| **核函数守卫** | `if asc.ascend_is_aic():` | 不需要 |
| **set_tail调用** | 传入3个参数 `(tail_m, tail_n, k_a)` | 传入2个参数 `(tail_m, tail_n)`，tail_k使用默认值-1 |
| **Bias支持** | ✅ | ✅ |
| **融合算子** | ❌（无Vector计算） | ✅（可接leaky_relu等） |
| **适用场景** | 纯Matmul | Matmul + Vector融合 |

> **核数差异的原因**：Cube Only模式只使用AIC核，每个AIC核独立处理一个分片，所以核数不除以2。MIX模式需要AIC+AIV成对工作，实际启动的核组数为总核数的一半。

---
# 4. Cube Only模式代码实现

实现一个带Bias的Cube Only多核Matmul算子。

**计算规格**：A[128,64] fp16 × B[64,30720] fp16 + Bias[1,30720] fp32 → C[128,30720] fp32，24核并行。

In [ ]:
%%writefile Sources/04.04/matmul_cube_only.py
# Copyright (c) 2025 Huawei Technologies Co., Ltd.
# CANN Open Software License Agreement Version 2.0

from typing import Tuple
import logging
import argparse
import torch

try:
    import torch_npu
except ModuleNotFoundError:
    pass

import asc
import asc.runtime.config as config
import asc.lib.runtime as rt
import asc.lib.host as host

logging.basicConfig(level=logging.INFO)

USE_CORE_NUM = 24
IS_TRANS_A = False
IS_TRANS_B = False
ENABLE_BIAS = True


@asc.jit(matmul_cube_only=True, always_compile=True)
def matmul_cube_only_kernel(a: asc.GlobalAddress, b: asc.GlobalAddress, c: asc.GlobalAddress,
                             bias: asc.GlobalAddress, tiling: asc.adv.TCubeTiling,
                             workspace: asc.GlobalAddress):
    # Cube Only模式：仅在AIC核上执行
    if asc.ascend_is_aic():
        # 计算多核偏移（含Bias偏移）
        offset_a, offset_b, offset_c, offset_bias, tail_m, tail_n = calc_offsets(
            tiling, IS_TRANS_A, IS_TRANS_B)

        # 绑定GM地址（带偏移）
        a_global = asc.GlobalTensor()
        b_global = asc.GlobalTensor()
        c_global = asc.GlobalTensor()
        bias_global = asc.GlobalTensor()
        a_global.set_global_buffer(a + offset_a)
        b_global.set_global_buffer(b + offset_b)
        c_global.set_global_buffer(c + offset_c)
        bias_global.set_global_buffer(bias + offset_bias)

        # 创建Matmul对象（含Bias类型）
        pipe = asc.TPipe()
        matmul = asc.adv.Matmul(
            a=asc.adv.MatmulType(asc.TPosition.GM, asc.CubeFormat.ND, a_global.dtype, IS_TRANS_A),
            b=asc.adv.MatmulType(asc.TPosition.GM, asc.CubeFormat.ND, b_global.dtype, IS_TRANS_B),
            c=asc.adv.MatmulType(asc.TPosition.GM, asc.CubeFormat.ND, c_global.dtype),
            bias=asc.adv.MatmulType(asc.TPosition.GM, asc.CubeFormat.ND, bias_global.dtype),
        )
        asc.adv.register_matmul(pipe, workspace, matmul, tiling)

        # 执行计算
        if asc.get_block_idx() < tiling.used_core_num:
            matmul.set_tensor_a(a_global, IS_TRANS_A)
            matmul.set_tensor_b(b_global, IS_TRANS_B)
            if tiling.is_bias:
                matmul.set_bias(bias_global)
            # Cube Only模式传入3个参数（tail_k=tiling.k_a）
            matmul.set_tail(tail_m, tail_n, tiling.k_a)
            matmul.iterate_all(c_global)
            matmul.end()

        asc.pipe_barrier(asc.PipeID.PIPE_ALL)


@asc.jit
def calc_offsets(tiling: asc.adv.TCubeTiling, is_trans_a: bool = False,
                 is_trans_b: bool = False) -> Tuple[int, int, int, int, int, int]:
    block_idx = asc.get_block_idx()
    m_single_blocks = tiling.m.ceildiv(tiling.single_core_m)
    n_index = block_idx // m_single_blocks
    m_index = block_idx % m_single_blocks
    offset_a = m_index * tiling.k_a * tiling.single_core_m
    if is_trans_a:
        offset_a = m_index * tiling.single_core_m
    offset_b = n_index * tiling.single_core_n
    if is_trans_b:
        offset_b = n_index * tiling.k_b * tiling.single_core_n
    offset_c = m_index * tiling.n * tiling.single_core_m + n_index * tiling.single_core_n
    offset_bias = n_index * tiling.single_core_n
    # 尾块处理
    tail_m = tiling.m - m_index * tiling.single_core_m
    if tail_m <= 0 or tail_m >= tiling.single_core_m:
        tail_m = tiling.single_core_m
    tail_n = tiling.n - n_index * tiling.single_core_n
    if tail_n <= 0 or tail_n >= tiling.single_core_n:
        tail_n = tiling.single_core_n
    return offset_a, offset_b, offset_c, offset_bias, tail_m, tail_n


def matmul_cube_only_launch(a: torch.Tensor, b: torch.Tensor, bias: torch.Tensor,
                             tiling: asc.adv.TCubeTiling, device) -> torch.Tensor:
    size_m, size_k = a.shape
    _, size_n = b.shape
    c = torch.zeros((size_m, size_n), dtype=torch.float32, device=device)
    workspace = torch.zeros(16 * 1024 * 1024, dtype=torch.uint8, device=device)
    # Cube Only模式：核数不除以2
    matmul_cube_only_kernel[tiling.used_core_num, rt.current_stream()](a, b, c, bias, tiling, workspace)
    return c


def generate_tiling(m, n, k) -> asc.adv.TCubeTiling:
    matmul_tiling = host.MultiCoreMatmulTiling(host.get_ascendc_platform())
    matmul_tiling.set_a_type(host.TPosition.GM, host.CubeFormat.ND, host.DataType.DT_FLOAT16, IS_TRANS_A)
    matmul_tiling.set_b_type(host.TPosition.GM, host.CubeFormat.ND, host.DataType.DT_FLOAT16, IS_TRANS_B)
    matmul_tiling.set_c_type(host.TPosition.GM, host.CubeFormat.ND, host.DataType.DT_FLOAT)
    matmul_tiling.set_bias_type(host.TPosition.GM, host.CubeFormat.ND, host.DataType.DT_FLOAT)
    matmul_tiling.set_dim(USE_CORE_NUM)
    matmul_tiling.set_org_shape(m, n, k)
    matmul_tiling.set_shape(m, n, k)
    matmul_tiling.enable_bias(ENABLE_BIAS)
    matmul_tiling.set_buffer_space(-1, -1, -1)

    tiling = asc.adv.TCubeTiling()
    matmul_tiling.get_tiling(tiling)
    return tiling


def matmul_cube_only_custom(backend: config.Backend, platform: config.Platform):
    config.set_platform(backend, platform)
    device = "npu" if config.Backend(backend) == config.Backend.NPU else "cpu"
    m, k, n = 128, 64, 30720
    a = torch.randint(-5, 5, (m, k), device=device).to(torch.float16)
    b = torch.randint(-5, 5, (k, n), device=device).to(torch.float16)
    bias = torch.randint(-5, 5, (1, n), device=device).to(torch.float32)

    if ENABLE_BIAS:
        golden = (torch.matmul(a.to(torch.float32), b.to(torch.float32)) + bias).to(torch.float32)
    else:
        golden = torch.matmul(a.to(torch.float32), b.to(torch.float32))

    tiling = generate_tiling(m, n, k)
    c = matmul_cube_only_launch(a, b, bias, tiling, device)
    assert torch.allclose(c, golden, rtol=1e-3, atol=1e-3)
    logging.info(f"[INFO] Cube Only模式验证通过! M={m}, N={n}, K={k}, 核数={USE_CORE_NUM}, Bias={ENABLE_BIAS}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("-r", type=str, default="NPU", help="backend to run")
    parser.add_argument("-v", type=str, default=None, help="platform to run")
    args = parser.parse_args()
    backend = args.r
    platform = args.v
    if backend not in config.Backend.__members__:
        raise ValueError(f"Unsupported Backend! Supported: {list(config.Backend.__members__.keys())}")
    backend = config.Backend(backend)
    if platform is not None:
        platform_values = [p.value for p in config.Platform]
        if platform not in platform_values:
            raise ValueError(f"Unsupported Platform! Supported: {platform_values}")
        platform = config.Platform(platform)
    logging.info("[INFO] start process sample matmul_cube_only.")
    matmul_cube_only_custom(backend, platform)
    logging.info("[INFO] Sample matmul_cube_only run success.")

In [ ]:
# 运行Cube Only示例（NPU模式）
!python3 Sources/04.04/matmul_cube_only.py -r NPU

---
# 5. MIX模式代码实现

实现一个MIX模式的多核Matmul算子（无Bias）。

**计算规格**：A[512,512] fp16 × B[512,1024] fp16 → C[512,1024] fp32，48核并行。

In [ ]:
%%writefile Sources/04.04/matmul_mix.py
# Copyright (c) 2025 Huawei Technologies Co., Ltd.
# CANN Open Software License Agreement Version 2.0

from typing import Tuple
import logging
import argparse
import torch

try:
    import torch_npu
except ModuleNotFoundError:
    pass

import asc
import asc.runtime.config as config
import asc.lib.runtime as rt
import asc.lib.host as host

logging.basicConfig(level=logging.INFO)

USE_CORE_NUM = 48
IS_TRANS_A = False
IS_TRANS_B = False


@asc.jit(always_compile=True)
def matmul_mix_kernel(a: asc.GlobalAddress, b: asc.GlobalAddress, c: asc.GlobalAddress,
                      tiling: asc.adv.TCubeTiling, workspace: asc.GlobalAddress):
    # MIX模式：计算多核偏移（无Bias偏移）
    offset_a, offset_b, offset_c, tail_m, tail_n = calc_offsets(tiling, IS_TRANS_A, IS_TRANS_B)

    # 绑定GM地址（带偏移）
    a_global = asc.GlobalTensor()
    b_global = asc.GlobalTensor()
    c_global = asc.GlobalTensor()
    a_global.set_global_buffer(a + offset_a)
    b_global.set_global_buffer(b + offset_b)
    c_global.set_global_buffer(c + offset_c)

    # 创建Matmul对象（无Bias）
    pipe = asc.TPipe()
    matmul = asc.adv.Matmul(
        a=asc.adv.MatmulType(asc.TPosition.GM, asc.CubeFormat.ND, a_global.dtype, IS_TRANS_A),
        b=asc.adv.MatmulType(asc.TPosition.GM, asc.CubeFormat.ND, b_global.dtype, IS_TRANS_B),
        c=asc.adv.MatmulType(asc.TPosition.GM, asc.CubeFormat.ND, c_global.dtype),
    )
    asc.adv.register_matmul(pipe, workspace, matmul, tiling)

    # 执行计算
    if asc.get_block_idx() < tiling.used_core_num:
        matmul.set_tensor_a(a_global, IS_TRANS_A)
        matmul.set_tensor_b(b_global, IS_TRANS_B)
        # MIX模式传入2个参数（tail_k使用默认值-1）
        matmul.set_tail(tail_m, tail_n)
        matmul.iterate_all(c_global)
        matmul.end()

    asc.pipe_barrier(asc.PipeID.PIPE_ALL)


@asc.jit
def calc_offsets(tiling: asc.adv.TCubeTiling,
                 is_trans_a: bool = False, is_trans_b: bool = False) -> Tuple[int, int, int, int, int]:
    block_idx = asc.get_block_idx()
    m_single_blocks = tiling.m.ceildiv(tiling.single_core_m)
    m_index = block_idx % m_single_blocks
    n_index = block_idx // m_single_blocks
    offset_a = m_index * tiling.k_a * tiling.single_core_m
    if is_trans_a:
        offset_a = m_index * tiling.single_core_m
    offset_b = n_index * tiling.single_core_n
    if is_trans_b:
        offset_b = n_index * tiling.k_b * tiling.single_core_n
    offset_c = m_index * tiling.n * tiling.single_core_m + n_index * tiling.single_core_n
    # 尾块处理
    tail_m = tiling.m - m_index * tiling.single_core_m
    if tail_m >= tiling.single_core_m:
        tail_m = tiling.single_core_m
    tail_n = tiling.n - n_index * tiling.single_core_n
    if tail_n >= tiling.single_core_n:
        tail_n = tiling.single_core_n
    return offset_a, offset_b, offset_c, tail_m, tail_n


def matmul_mix_launch(a: torch.Tensor, b: torch.Tensor,
                      tiling: asc.adv.TCubeTiling, device) -> torch.Tensor:
    size_m, _ = a.shape
    _, size_n = b.shape
    c = torch.zeros((size_m, size_n), dtype=torch.float32, device=device)
    workspace = torch.zeros(16 * 1024 * 1024, dtype=torch.uint8, device=device)
    # MIX模式：核数除以2（AIC+AIV成对）
    matmul_mix_kernel[USE_CORE_NUM // 2, rt.current_stream()](a, b, c, tiling, workspace)
    return c


def generate_tiling(m, n, k):
    matmul_tiling = host.MultiCoreMatmulTiling(host.get_ascendc_platform())
    matmul_tiling.set_a_type(host.TPosition.GM, host.CubeFormat.ND, host.DataType.DT_FLOAT16, IS_TRANS_A)
    matmul_tiling.set_b_type(host.TPosition.GM, host.CubeFormat.ND, host.DataType.DT_FLOAT16, IS_TRANS_B)
    matmul_tiling.set_c_type(host.TPosition.GM, host.CubeFormat.ND, host.DataType.DT_FLOAT)
    matmul_tiling.set_bias_type(host.TPosition.GM, host.CubeFormat.ND, host.DataType.DT_FLOAT)
    matmul_tiling.set_dim(USE_CORE_NUM)
    matmul_tiling.set_org_shape(m, n, k)
    matmul_tiling.set_shape(m, n, k)
    matmul_tiling.enable_bias(False)
    matmul_tiling.set_buffer_space(-1, -1, -1)

    tiling = asc.adv.TCubeTiling()
    matmul_tiling.get_tiling(tiling)
    return tiling


def matmul_mix_custom(backend: config.Backend, platform: config.Platform):
    config.set_platform(backend, platform)
    device = "npu" if config.Backend(backend) == config.Backend.NPU else "cpu"
    m, k, n = 512, 512, 1024
    a = torch.randint(-5, 5, (m, k), device=device).to(torch.float16)
    b = torch.randint(-5, 5, (k, n), device=device).to(torch.float16)
    golden = torch.matmul(a.to(torch.float32), b.to(torch.float32))

    tiling = generate_tiling(m, n, k)
    c = matmul_mix_launch(a, b, tiling, device)
    assert torch.allclose(c, golden, rtol=1e-3, atol=1e-3)
    logging.info(f"[INFO] MIX模式验证通过! M={m}, N={n}, K={k}, 核数={USE_CORE_NUM}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("-r", type=str, default="NPU", help="backend to run")
    parser.add_argument("-v", type=str, default=None, help="platform to run")
    args = parser.parse_args()
    backend = args.r
    platform = args.v
    if backend not in config.Backend.__members__:
        raise ValueError(f"Unsupported Backend! Supported: {list(config.Backend.__members__.keys())}")
    backend = config.Backend(backend)
    if platform is not None:
        platform_values = [p.value for p in config.Platform]
        if platform not in platform_values:
            raise ValueError(f"Unsupported Platform! Supported: {platform_values}")
        platform = config.Platform(platform)
    logging.info("[INFO] start process sample matmul_mix.")
    matmul_mix_custom(backend, platform)
    logging.info("[INFO] Sample matmul_mix run success.")

In [ ]:
# 运行MIX模式示例（NPU模式）
!python3 Sources/04.04/matmul_mix.py -r NPU

---
# 6. 多核偏移计算详解

`calc_offsets`是多核并行的核心函数，它根据当前核的`block_idx`计算该核负责的数据分片在GM中的偏移地址。

### 偏移计算原理

矩阵C[M, N]被切分为多个`single_core_m × single_core_n`的分片，按"先M后N"的顺序分配给各核：

```
         N轴
    ┌──────┬──────┬──────┐
  M │核0   │核3   │核6   │
  轴│      │      │      │
    ├──────┼──────┼──────┤
    │核1   │核4   │核7   │
    │      │      │      │
    ├──────┼──────┼──────┤
    │核2   │核5   │核8   │
    │      │      │      │
    └──────┴──────┴──────┘

m_single_blocks = ceildiv(M, single_core_m) = 3
核0: m_index=0, n_index=0
核4: m_index=1, n_index=1
```

<img src="./images/04.04_cube_only_and_mix_mode_matmul/multi_core_tiling.png" alt="多核分块示意图" width="400px">

### 代码解析

```python
block_idx = asc.get_block_idx()                           # 当前核编号
m_single_blocks = tiling.m.ceildiv(tiling.single_core_m)   # M方向分块数
m_index = block_idx % m_single_blocks                      # M方向索引（先M）
n_index = block_idx // m_single_blocks                     # N方向索引（后N）

# A矩阵偏移：M方向移动，每个分片大小为 single_core_m × k_a
offset_a = m_index * tiling.k_a * tiling.single_core_m

# B矩阵偏移：N方向移动，每个分片大小为 single_core_n
offset_b = n_index * tiling.single_core_n

# C矩阵偏移：M和N方向都有，行优先
offset_c = m_index * tiling.n * tiling.single_core_m + n_index * tiling.single_core_n

# 尾块：处理不能整除的情况
tail_m = tiling.m - m_index * tiling.single_core_m
if tail_m >= tiling.single_core_m:
    tail_m = tiling.single_core_m     # 满块
```

### 转置处理

当`is_trans_a=True`时，A矩阵在内存中按`[K, M]`排布，偏移计算变为：
```python
offset_a = m_index * tiling.single_core_m  # 不乘k_a
```

---
# 7. Bias处理与代码解析

### 与04.03高阶API的关键差异

| 差异点 | 04.03 高阶API | Cube Only | MIX |
| --- | --- | --- | --- |
| 装饰器 | `@asc.jit(always_compile=True)` | `matmul_cube_only=True` | 默认 |
| 核函数守卫 | 无 | `if asc.ascend_is_aic():` | 无 |
| calc_offsets返回值 | 5个（无offset_bias） | 6个（含offset_bias） | 5个 |
| set_tail | 传入2个参数 | 传入3个参数 | 传入2个参数 |
| set_bias | 无 | `matmul.set_bias(bias_global)` | 无（可扩展） |
| Launch核数 | `USE_CORE_NUM // 2` | `tiling.used_core_num` | `USE_CORE_NUM // 2` |

### Bias处理流程

1. **Tiling侧**：`matmul_tiling.enable_bias(True)` + `set_bias_type(...)`
2. **Kernel侧**：Matmul对象声明 `bias=MatmulType(...)` + `matmul.set_bias(bias_global)`
3. **偏移计算**：`offset_bias = n_index * tiling.single_core_n`（Bias按N轴分片）

下图展示了Bias从配置到执行的完整流程：Tiling侧声明启用 → Kernel侧创建含Bias的Matmul对象 → 多核偏移按N轴分片 → iterate时自动将Bias加到L0C结果上。

<img src="./images/04.04_cube_only_and_mix_mode_matmul/bias_processing_flow.png" alt="Bias处理流程" width="700px">

### 尾块处理差异

Cube Only模式的尾块判断多了 `tail_m <= 0` 的条件：
```python
if tail_m <= 0 or tail_m >= tiling.single_core_m:
    tail_m = tiling.single_core_m
```
这是因为Cube Only模式下核数映射方式不同，可能出现负值的情况。

---
# 8. Cube Only vs MIX选型

### 选型决策表

| 场景 | 推荐模式 | 原因 |
| --- | --- | --- |
| 纯Matmul（无融合） | Cube Only | AIC核利用率最大化，核数不除以2 |
| Matmul + Vector融合 | MIX | 需要AIV核执行Vector计算 |
| 追求最大Cube吞吐 | Cube Only | 独占所有AIC核 |
| 需要后处理（量化/ReLU等） | MIX | AIV核可执行后处理 |
| 简单原型验证 | MIX | 默认模式，代码更简洁 |

### 性能对比要点

- **Cube Only**：核数 = `used_core_num`，每个AIC核独立处理分片
- **MIX**：核组数 = `used_core_num // 2`，每个核组包含AIC+AIV
- 对于纯Matmul，Cube Only理论上可获得更高的Cube利用率
- MIX的优势在于融合算子减少数据搬运

---
# 9. 小结

### 四种模式对比总览

| 维度 | 基础API (04.02) | 高阶API (04.03) | Cube Only (04.04) | MIX (04.04) |
| --- | --- | --- | --- | --- |
| API层级 | load_data/mmad/fixpipe | asc.adv.Matmul | asc.adv.Matmul | asc.adv.Matmul |
| K方向分块 | 手动for循环 | ✅ 自动 | ✅ 自动 | ✅ 自动 |
| 多核并行 | ❌ | ✅ | ✅ | ✅ |
| Bias | ❌ | ❌ | ✅ | ❌（可扩展） |
| 融合算子 | ❌ | ❌ | ❌ | ✅（第5章） |
| 核数 | 1 | `//2` | 不除2 | `//2` |

---

## 课后练习

### 选择题

**1.** 启用Cube Only模式需要设置哪个装饰器参数？

- A. `@asc.jit(cube=True)`
- B. `@asc.jit(matmul_cube_only=True)`
- C. `@asc.jit(aic_only=True)`
- D. `@asc.jit(no_vector=True)`

**2.** Cube Only模式下kernel launch的核数是？

- A. `USE_CORE_NUM // 2`
- B. `tiling.used_core_num`
- C. `USE_CORE_NUM * 2`
- D. `1`

**3.** MIX模式下kernel launch的核数为什么是 `USE_CORE_NUM // 2`？

- A. 因为只有一半的核可用
- B. 因为AIC+AIV成对工作，每个核组包含1个AIC和1个AIV
- C. 因为硬件限制
- D. 为了降低功耗

**4.** `calc_offsets`中 `m_index` 的计算方式是？

- A. `block_idx // m_single_blocks`
- B. `block_idx % m_single_blocks`
- C. `block_idx * m_single_blocks`
- D. `tiling.m // tiling.single_core_m`

### 填空题

**5.** 在Cube Only模式下，核函数内的计算逻辑需要用 `if ______ :` 包裹，确保仅在AIC核上执行。

**6.** 在Tiling配置中调用 ______ 启用Bias，在kernel中调用 ______ 设置偏置张量。Cube Only模式下set_tail需传入 ______ 个参数。

**7.** 尾块处理的目的是什么？当 `tail_m >= tiling.single_core_m` 时应设为什么值？

**8.** 当 `is_trans_a=True` 时，A矩阵的偏移计算从 `m_index * tiling.k_a * tiling.single_core_m` 变为 ______ 。

---

> 点击下方查看答案

In [ ]:
!cat ./answer/04.04_cube_only_and_mix_mode_matmul/practice_answers.md